In [15]:
# Install required packages for GlotLID
# !pip install transformers torch fasttext huggingface_hub

from huggingface_hub import hf_hub_download
import fasttext
import os

# Download GlotLID model from Hugging Face
# GlotLID is a FastText-based language identification model supporting 1600+ languages
MODEL_PATH = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")

# Load the model
glotlid_model = fasttext.load_model(MODEL_PATH)

print(f"GlotLID model loaded successfully!")
print(f"Model path: {MODEL_PATH}")

# Test the model with sample texts
test_texts = [
    "Bonjour, comment allez-vous?",  # French
    "Hello, how are you today?",      # English
    "Azul fellawen, amek tellam?",    # Kabyle
]

print("\nTesting GlotLID predictions:")
for text in test_texts:
    # predict returns (labels, probabilities)
    labels, probs = glotlid_model.predict(text.replace('\n', ' '), k=3)
    print(f"\nText: '{text}'")
    for label, prob in zip(labels, probs):
        lang_code = label.replace('__label__', '')
        print(f"  {lang_code}: {prob:.2%}")

GlotLID model loaded successfully!
Model path: /teamspace/studios/this_studio/.cache/huggingface/hub/models--cis-lmu--glotlid/snapshots/74cb50b709c9eefe0f790030c6c95c461b4e3b77/model.bin

Testing GlotLID predictions:

Text: 'Bonjour, comment allez-vous?'
  fra_Latn: 94.62%
  pcd_Latn: 3.79%
  eng_Latn: 0.49%

Text: 'Hello, how are you today?'
  eng_Latn: 100.00%
  ind_Latn: 0.00%
  som_Latn: 0.00%

Text: 'Azul fellawen, amek tellam?'
  kab_Latn: 100.00%
  taq_Latn: 0.00%
  thv_Latn: 0.00%


In [19]:
import re

# Input/Output files
INPUT_FILE = r"en_kab_pairs.txt"

# Separator pattern: one or more tabs
TAB_PATTERN = re.compile(r'\t+')

# Language codes to filter out (French and English)
LANGUAGES_TO_REMOVE = {
    'fra_Latn',  # French (Latin script)
    'eng_Latn',  # English (Latin script)
}

# Kabyle language code in GlotLID
KABYLE_CODE = 'kab_Latn'


def detect_language(text: str, model, top_k: int = 1) -> list:
    """
    Detect language of text using GlotLID.

    Returns:
        list of tuples: [(language_code, probability), ...]
    """
    # Clean text for prediction (remove newlines)
    clean_text = text.replace('\n', ' ').strip()

    if not clean_text:
        return [('unknown', 0.0)]

    labels, probs = model.predict(clean_text, k=top_k)

    results = []
    for label, prob in zip(labels, probs):
        lang_code = label.replace('__label__', '')
        results.append((lang_code, float(prob)))

    return results


def is_foreign_language(text: str, model, foreign_langs: set,
                        confidence_threshold: float = 0.5) -> tuple:
    """
    Check if text is detected as a foreign language (French/English).

    Returns:
        tuple: (is_foreign, detected_lang, confidence)
    """
    predictions = detect_language(text, model, top_k=1)

    if not predictions:
        return False, 'unknown', 0.0

    top_lang, top_prob = predictions[0]

    # Check if the detected language is in our list of foreign languages
    # and the confidence is above threshold
    is_foreign = top_lang in foreign_langs and top_prob >= confidence_threshold

    return is_foreign, top_lang, top_prob


# Test the function with sample sentences
test_sentences = [
    "Yusa-d aṭas n tikkal.",           # Good Kabyle
    "La tettru yemma-s.",              # Good Kabyle (la = progressive marker)
    "Ass-a d ass ameqqran.",           # Good Kabyle
    "Je suis content de te voir.",     # French
    "The house is very beautiful.",    # English
    "Azul, comment ça va mon ami?",    # Mixed Kabyle/French
    "Axxam-is est très grand.",        # Mixed - more French
    "Nekk d aqvayli.",                 # Kabyle
]

print("Testing GlotLID language detection:\n")
print("-" * 80)
for sent in test_sentences:
    is_foreign, lang, conf = is_foreign_language(
        sent, glotlid_model, LANGUAGES_TO_REMOVE, confidence_threshold=0.5
    )
    status = "❌ REMOVE" if is_foreign else "✅ KEEP"
    print(f"{status} | Lang: {lang:12} | Conf: {conf:.1%} | Text: '{sent}'")

Testing GlotLID language detection:

--------------------------------------------------------------------------------
✅ KEEP | Lang: kab_Latn     | Conf: 100.0% | Text: 'Yusa-d aṭas n tikkal.'
✅ KEEP | Lang: kab_Latn     | Conf: 100.0% | Text: 'La tettru yemma-s.'
✅ KEEP | Lang: kab_Latn     | Conf: 100.0% | Text: 'Ass-a d ass ameqqran.'
❌ REMOVE | Lang: fra_Latn     | Conf: 99.8% | Text: 'Je suis content de te voir.'
❌ REMOVE | Lang: eng_Latn     | Conf: 100.0% | Text: 'The house is very beautiful.'
❌ REMOVE | Lang: fra_Latn     | Conf: 99.9% | Text: 'Azul, comment ça va mon ami?'
❌ REMOVE | Lang: fra_Latn     | Conf: 93.5% | Text: 'Axxam-is est très grand.'
✅ KEEP | Lang: kab_Latn     | Conf: 100.0% | Text: 'Nekk d aqvayli.'


In [20]:
from tqdm import tqdm

# Configuration for GlotLID-based language filtering
CONFIDENCE_THRESHOLD = 0.5  # Minimum confidence to consider a detection valid
# Don't remove short sentences (GlotLID less reliable on short text)
MIN_CHARS_TO_KEEP = 20

OUTPUT_FILE = r"cleaned_kab_pairs.txt"
REMOVED_FILE = r"removed_foreign_pairs.txt"


def count_lines(filepath: str) -> int:
    """Count total lines in a file for progress bar."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def filter_by_glotlid(input_file: str, output_file: str, removed_file: str = None,
                      model=None, foreign_langs: set = None,
                      confidence_threshold: float = 0.5,
                      min_chars_to_keep: int = 20):
    """
    Filter pairs where the Kabyle sentence is detected as French/English by GlotLID.

    Args:
        input_file: Path to input file with tab-separated pairs
        output_file: Path to save cleaned pairs
        removed_file: Path to save removed pairs (optional)
        model: GlotLID model
        foreign_langs: Set of language codes to filter out
        confidence_threshold: Minimum confidence for language detection
        min_chars_to_keep: Short sentences below this are always kept
    """
    # Count total lines for progress bar
    print("Counting lines...")
    total_lines_count = count_lines(input_file)
    print(f"Total lines to process: {total_lines_count:,}")

    total_lines = 0
    kept_lines = 0
    removed_lines = 0
    malformed_lines = 0

    # Track detected languages for statistics
    lang_stats = {}

    fout = open(output_file, 'w', encoding='utf-8')
    frem = open(removed_file, 'w', encoding='utf-8') if removed_file else None

    try:
        with open(input_file, 'r', encoding='utf-8') as fin:
            # Create progress bar
            pbar = tqdm(fin, total=total_lines_count,
                        desc="Filtering", unit=" lines",
                        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')

            for line in pbar:
                total_lines += 1

                line = line.strip()
                parts = TAB_PATTERN.split(line)

                if len(parts) < 2:
                    malformed_lines += 1
                    continue

                source_text = parts[0].strip()
                kab_text = parts[1].strip()

                # Get language detection for statistics
                predictions = detect_language(kab_text, model, top_k=1)
                detected_lang = predictions[0][0] if predictions else 'unknown'
                lang_stats[detected_lang] = lang_stats.get(
                    detected_lang, 0) + 1

                # Exception: keep short sentences (GlotLID less reliable)
                if len(kab_text) < min_chars_to_keep:
                    fout.write(f"{source_text}\t{kab_text}\n")
                    kept_lines += 1
                    continue

                # Check if detected as foreign language
                is_foreign, lang, conf = is_foreign_language(
                    kab_text, model, foreign_langs, confidence_threshold
                )

                if not is_foreign:
                    fout.write(f"{source_text}\t{kab_text}\n")
                    kept_lines += 1
                else:
                    removed_lines += 1
                    if frem:
                        frem.write(
                            f"{source_text}\t{kab_text}\t{lang}\t{conf:.2%}\n")

                # Update progress bar description with stats
                if total_lines % 1000 == 0:
                    keep_pct = (kept_lines / total_lines *
                                100) if total_lines > 0 else 0
                    pbar.set_postfix(
                        {'kept': f'{keep_pct:.1f}%', 'removed': removed_lines})

    finally:
        fout.close()
        if frem:
            frem.close()

    return {
        'total': total_lines,
        'kept': kept_lines,
        'removed': removed_lines,
        'malformed': malformed_lines,
        'kept_percentage': (kept_lines / total_lines * 100) if total_lines > 0 else 0,
        'lang_stats': lang_stats
    }


print("GlotLID Language Filter Configuration:")
print("=" * 50)
print(f"  Languages to remove: {LANGUAGES_TO_REMOVE}")
print(f"  Confidence threshold: {CONFIDENCE_THRESHOLD:.0%}")
print(f"  Min chars to keep (exception): {MIN_CHARS_TO_KEEP}")
print(f"  Input file: {INPUT_FILE}")
print(f"  Cleaned output: {OUTPUT_FILE}")
print(f"  Removed pairs: {REMOVED_FILE}")

GlotLID Language Filter Configuration:
  Languages to remove: {'eng_Latn', 'fra_Latn'}
  Confidence threshold: 50%
  Min chars to keep (exception): 20
  Input file: en_kab_pairs.txt
  Cleaned output: cleaned_kab_pairs.txt
  Removed pairs: removed_foreign_pairs.txt


In [21]:
# ⚠️ RUN THIS CELL TO FILTER BY LANGUAGE USING GLOTLID
# This will remove pairs where Kabyle text is detected as French/English

print("Starting GlotLID language filter...")
print("=" * 60)
print(f"  Input: {INPUT_FILE}")
print(f"  Languages to remove: {LANGUAGES_TO_REMOVE}")
print(f"  Confidence threshold: {CONFIDENCE_THRESHOLD:.0%}")
print(f"  Min chars to keep: {MIN_CHARS_TO_KEEP}")
print(f"  Cleaned output: {OUTPUT_FILE}")
print(f"  Removed pairs: {REMOVED_FILE}")
print("=" * 60)

stats = filter_by_glotlid(
    INPUT_FILE,
    OUTPUT_FILE,
    removed_file=REMOVED_FILE,
    model=glotlid_model,
    foreign_langs=LANGUAGES_TO_REMOVE,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    min_chars_to_keep=MIN_CHARS_TO_KEEP
)

print("\n" + "=" * 60)
print("GLOTLID LANGUAGE FILTERING COMPLETE")
print("=" * 60)
print(f"Total pairs processed: {stats['total']:,}")
print(
    f"Pairs kept:           {stats['kept']:,} ({stats['kept_percentage']:.1f}%)")
print(
    f"Pairs removed:        {stats['removed']:,} ({100-stats['kept_percentage']:.1f}%)")
print(f"Malformed lines:      {stats['malformed']:,}")
print(f"\nCleaned pairs saved to: {OUTPUT_FILE}")
print(f"Removed pairs saved to: {REMOVED_FILE}")

Starting GlotLID language filter...
  Input: en_kab_pairs.txt
  Languages to remove: {'eng_Latn', 'fra_Latn'}
  Confidence threshold: 50%
  Min chars to keep: 20
  Cleaned output: cleaned_kab_pairs.txt
  Removed pairs: removed_foreign_pairs.txt
Counting lines...


Total lines to process: 4,088,851


Filtering:  43%|████▎     | 1738642/4088851 [33:58<45:55, 853.07 lines/s]  


KeyboardInterrupt: 

In [ ]:
# Display language detection statistics
print("Language Detection Statistics:")
print("=" * 50)

# Sort by count (descending)
sorted_langs = sorted(stats['lang_stats'].items(),
                      key=lambda x: x[1], reverse=True)

print(f"\n{'Language Code':<20} {'Count':>10} {'Percentage':>12}")
print("-" * 45)

for lang, count in sorted_langs[:20]:  # Show top 20 languages
    pct = count / stats['total'] * 100
    print(f"{lang:<20} {count:>10,} {pct:>11.1f}%")

if len(sorted_langs) > 20:
    print(f"\n... and {len(sorted_langs) - 20} more languages detected")

In [ ]:
# Preview some removed pairs to verify the filtering
print("Sample of removed pairs (detected as French/English):")
print("=" * 80)

with open(REMOVED_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10:  # Show first 10 removed pairs
            break
        parts = line.strip().split('\t')
        if len(parts) >= 4:
            source, kab, lang, conf = parts[0], parts[1], parts[2], parts[3]
            print(f"\n[{i+1}] Detected: {lang} ({conf})")
            print(f"    Source: {source[:60]}...")
            print(f"    Kab:    {kab[:60]}...")